# 11 - ETH Options (separate DB)

Options data lives in `database/options.db`, not `market.db`.

Covers:

1. Load option_chain
2. IV smile (one expiry)
3. IV surface (strike × expiry)
4. Put/call ratio over time
5. ATM IV vs realized vol
6. Skew & term structure
7. Summary dashboard


## Setup

In [ ]:
import sys, sqlite3
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

_p = Path.cwd().resolve()
while _p != _p.parent and not (_p / "database").exists():
    _p = _p.parent
sys.path.insert(0, str(_p))

OPT_DB  = _p / "database" / "options.db"
MKT_DB  = _p / "database" / "market.db"

pd.set_option("display.width", 180)
plt.rcParams["figure.figsize"] = (13, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

def opt(q, **p):
    with sqlite3.connect(OPT_DB) as c:
        return pd.read_sql_query(q, c, params=p)

def mkt(q, **p):
    with sqlite3.connect(MKT_DB) as c:
        return pd.read_sql_query(q, c, params=p)

print("options.db:", OPT_DB)
print("market.db :", MKT_DB)

## 1. Load option chain

In [ ]:
oc = opt("SELECT * FROM option_chain ORDER BY ts DESC, expiry, strike")
print("rows:", len(oc))
if not oc.empty:
    print("range:", pd.to_datetime(oc.ts.min(), unit="ms"), "->",
          pd.to_datetime(oc.ts.max(), unit="ms"))
    print("snapshots:", oc.ts.nunique())
    print("expiries :", oc.expiry.nunique())
oc.head()

## 2. IV smile — nearest expiry

In [ ]:
if not oc.empty:
    oc["dte"] = (oc.expiry - oc.ts) / 86_400_000.0
    valid = oc[(oc.dte > 2) & (oc.dte < 60) & oc.mark_iv.notna()]
    if not valid.empty:
        target = valid.expiry.value_counts().index[0]
        snap = valid[valid.expiry == target]
        latest = snap[snap.ts == snap.ts.max()]

        fig, ax = plt.subplots()
        for kind, color in [("C", "seagreen"), ("P", "crimson")]:
            s = latest[latest["kind"] == kind].sort_values("strike")
            ax.plot(s.strike, s.mark_iv * 100, "o-", color=color,
                    label="Calls" if kind == "C" else "Puts")
        ax.set_title(f"IV smile — {pd.to_datetime(target, unit='ms').date()}")
        ax.set_xlabel("strike")
        ax.set_ylabel("IV %")
        ax.legend()
        plt.tight_layout()
        plt.show()
    else:
        print("no snapshots yet")

## 3. IV surface — strike × expiry

In [ ]:
if not oc.empty and "dte" in oc.columns:
    surf = oc[oc.mark_iv.notna()].copy()
    surf = surf[(surf.dte > 0) & (surf.dte < 120)]
    if not surf.empty:
        latest_ts = surf.ts.max()
        snap = surf[surf.ts == latest_ts]
        pivot = snap.pivot_table(index="expiry", columns="strike",
                                 values="mark_iv", aggfunc="mean")
        if not pivot.empty:
            fig, ax = plt.subplots(figsize=(12, 6))
            im = ax.imshow(pivot.values * 100, aspect="auto", cmap="viridis")
            ax.set_yticks(range(len(pivot.index)))
            ax.set_yticklabels([pd.to_datetime(e, unit="ms").date() for e in pivot.index])
            ax.set_xticks(range(len(pivot.columns)))
            ax.set_xticklabels([f"{int(s)}" for s in pivot.columns], rotation=45)
            plt.colorbar(im, ax=ax, label="IV %")
            ax.set_title("IV surface")
            ax.set_xlabel("strike")
            ax.set_ylabel("expiry")
            plt.tight_layout()
            plt.show()

## 4. Put/call ratio over time

In [ ]:
sm = opt("SELECT * FROM option_summary WHERE underlying='ETH' ORDER BY ts")
if not sm.empty:
    sm["date"] = pd.to_datetime(sm.ts, unit="ms")
    sm = sm.set_index("date")

    fig, ax = plt.subplots()
    ax.plot(sm.index, sm.pcr_oi, label="PCR (OI)", color="steelblue")
    ax.axhline(1.0, ls="--", color="gray", lw=0.8)
    ax.set_title("ETH put/call ratio (OI)")
    ax.legend()
    plt.tight_layout()
    plt.show()
    print(sm[["pcr_oi", "pcr_vol"]].tail())
else:
    print("no summary rows yet")

## 5. ATM IV vs realized vol

In [ ]:
if not sm.empty:
    px = mkt("SELECT open_time, close FROM ohlcv WHERE symbol='ETHUSDT' AND timeframe='1d' ORDER BY open_time")
    px["date"] = pd.to_datetime(px.open_time, unit="ms")
    px = px.set_index("date")["close"]
    rv = px.pct_change().rolling(30).std() * np.sqrt(365)

    fig, ax = plt.subplots()
    if sm.atm_iv.notna().any():
        ax.plot(sm.index, sm.atm_iv * 100, label="ATM IV", color="crimson")
    ax.plot(rv.index, rv * 100, label="30d realized vol", color="steelblue")
    ax.set_title("vol comparison")
    ax.legend()
    plt.tight_layout()
    plt.show()

## 6. Summary dashboard

In [ ]:
if not sm.empty:
    last = sm.iloc[-1]
    print("=" * 50)
    print("  ETH OPTIONS SNAPSHOT")
    print("=" * 50)
    print(f"  as of     : {last.name}")
    print(f"  PCR (OI)  : {last.pcr_oi}")
    print(f"  PCR (vol) : {last.pcr_vol}")
    print(f"  max pain  : {last.max_pain}")
    print("=" * 50)

## Next steps

- Fetch hourly: `python -m scripts_python.fetch_options --currency ETH`
- Add to scheduler
- Merge into RL features later by joining on `ts`
